# Global Digital Readiness
## Economic and connectivity foundations of participation in the digital and AI economy

This notebook reproduces the analytical visualizations shown in the companion web application.

**Research question:** How are national income, population, internet use, broadband connectivity, and electricity access associated with countries' digital readiness?

**Important interpretation note:** These World Bank indicators describe economic and digital infrastructure conditions. They do **not** directly measure AI capability, AI adoption, AI investment, or AI research leadership. Any relationship with AI development is discussed as context or a hypothesis, not as a direct measurement.

### Data source
World Bank Indicators API: https://api.worldbank.org/

The notebook downloads live data when run. The latest available year can differ by indicator and country because World Bank reporting is not simultaneous.

## 1. Indicator catalogue

| Indicator | World Bank code | Meaning |
|---|---|---|
| GDP per capita (current US$) | `NY.GDP.PCAP.CD` | Average economic output per person, in current US dollars |
| Population, total | `SP.POP.TOTL` | Total population |
| Individuals using the Internet (% of population) | `IT.NET.USER.ZS` | Share of population using the Internet |
| Fixed broadband subscriptions (per 100 people) | `IT.NET.BBND.P2` | Fixed broadband subscriptions per 100 people |
| Access to electricity (% of population) | `EG.ELC.ACCS.ZS` | Population with access to electricity |

These are selected as complementary indicators: economic capacity, scale, internet access, broadband infrastructure, and basic electricity access.

In [ ]:
import requests
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, Markdown

BASE_URL = "https://api.worldbank.org/v2"
START_YEAR = 2000
END_YEAR = 2024

INDICATORS = {
    "gdp_per_capita": {
        "code": "NY.GDP.PCAP.CD",
        "label": "GDP per capita (current US$)",
        "unit": "current US$"
    },
    "population": {
        "code": "SP.POP.TOTL",
        "label": "Population, total",
        "unit": "people"
    },
    "internet_users": {
        "code": "IT.NET.USER.ZS",
        "label": "Individuals using the Internet (% of population)",
        "unit": "%"
    },
    "fixed_broadband": {
        "code": "IT.NET.BBND.P2",
        "label": "Fixed broadband subscriptions (per 100 people)",
        "unit": "subscriptions per 100 people"
    },
    "electricity_access": {
        "code": "EG.ELC.ACCS.ZS",
        "label": "Access to electricity (% of population)",
        "unit": "%"
    }
}

print("Configured indicators:", len(INDICATORS))

In [ ]:
def fetch_indicator(key, meta):
    url = f"{BASE_URL}/country/all/indicator/{meta['code']}"
    params = {
        "format": "json",
        "per_page": 20000,
        "date": f"{START_YEAR}:{END_YEAR}"
    }
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()

    if not isinstance(payload, list) or len(payload) < 2 or payload[1] is None:
        raise RuntimeError(f"No observations returned for {meta['code']}")

    rows = []
    for item in payload[1]:
        # World Bank regional/income aggregates have region.id == 'NA'.
        # Keep country/economy observations only.
        if not item.get("countryiso3code") or item.get("region", {}).get("id") == "NA":
            continue
        rows.append({
            "iso3": item.get("countryiso3code"),
            "country": item.get("country", {}).get("value"),
            "year": int(item["date"]),
            key: item.get("value")
        })
    return pd.DataFrame(rows)

frames = {}
for key, meta in INDICATORS.items():
    frames[key] = fetch_indicator(key, meta)
    print(f"{meta['code']}: {len(frames[key]):,} country-year observations")

In [ ]:
# Merge indicator series on country and year.
data = None
for key, frame in frames.items():
    data = frame if data is None else data.merge(
        frame, on=["iso3", "country", "year"], how="outer"
    )

data = data.sort_values(["country", "year"]).reset_index(drop=True)
numeric_cols = list(INDICATORS.keys())
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

print("Rows:", f"{len(data):,}")
print("Countries/economies:", data["iso3"].nunique())
print("Year range:", data["year"].min(), "to", data["year"].max())
display(data.head())

## 2. Data quality and coverage

Missing values are expected: countries report different indicators in different years. We do not replace missing observations with zero. The latest-year cross-country comparison uses the latest year available **for each indicator**, and the chart title/notes make that clear.

In [ ]:
coverage = (
    data[numeric_cols].notna().sum()
    .rename("non_missing_observations")
    .to_frame()
)
coverage["missing_pct"] = (1 - coverage["non_missing_observations"] / len(data)) * 100
display(coverage.round(1))

## 3. Shared chart dataset

The web app and notebook use the same definitions:
- `latest_by_indicator`: latest non-null observation per country for each indicator.
- `latest_common`: countries with all five indicators in the same year, used only where a same-year comparison is necessary.
- Time-series charts retain each country's actual reported year and do not interpolate missing values.

In [ ]:
def latest_snapshot(column):
    subset = data.dropna(subset=[column]).sort_values("year")
    return subset.groupby("iso3", as_index=False).tail(1).copy()

latest_internet = latest_snapshot("internet_users")
latest_gdp = latest_snapshot("gdp_per_capita")
latest_population = latest_snapshot("population")
latest_broadband = latest_snapshot("fixed_broadband")
latest_electricity = latest_snapshot("electricity_access")

# Build latest available country-level profile, potentially with different
# observation years by indicator. Year columns are retained for transparency.
latest_by_indicator = None
for key in numeric_cols:
    part = latest_snapshot(key)[["iso3", "country", "year", key]].rename(
        columns={"year": f"{key}_year"}
    )
    if latest_by_indicator is None:
        latest_by_indicator = part
    else:
        latest_by_indicator = latest_by_indicator.merge(
            part, on=["iso3", "country"], how="outer"
        )

latest_by_indicator.head()

## 4. Visualization 1 — Global internet access map

Shows the latest reported internet-use percentage for each country. Hover to inspect the country, value, and reporting year.

In [ ]:
fig_map = px.choropleth(
    latest_internet,
    locations="iso3",
    color="internet_users",
    hover_name="country",
    hover_data={"iso3": True, "internet_users": ":.1f", "year": True},
    color_continuous_scale="Blues",
    range_color=(0, 100),
    title="Internet use by country — latest available observation",
    labels={"internet_users": "Internet users (%)", "year": "Reported year"}
)
fig_map.update_layout(template="plotly_white", height=620, margin=dict(l=0,r=0,t=60,b=0))
fig_map.show()

## 5. Visualization 2 — Income and internet access

A cross-sectional scatter plot using the latest reported observation for each measure. Since the two indicators may have different reporting years, the year for each measure is included in the hover details. This is an exploratory association, not evidence of causation.

In [ ]:
scatter = latest_by_indicator.dropna(subset=["gdp_per_capita", "internet_users"]).copy()
fig_income = px.scatter(
    scatter,
    x="gdp_per_capita",
    y="internet_users",
    size="population",
    color="internet_users",
    hover_name="country",
    hover_data={
        "gdp_per_capita": ":$,.0f",
        "internet_users": ":.1f",
        "population": ":,.0f",
        "gdp_per_capita_year": True,
        "internet_users_year": True
    },
    log_x=True,
    color_continuous_scale="Teal",
    title="GDP per capita and internet use",
    labels={
        "gdp_per_capita": "GDP per capita (current US$, log scale)",
        "internet_users": "Internet users (%)",
        "population": "Population"
    }
)
fig_income.update_layout(template="plotly_white", height=560)
fig_income.show()

## 6. Visualization 2b — Countries with highest internet use

A ranked view of the countries with the highest latest reported internet-use percentage. The reported year is retained in the hover details.


In [ ]:
top_internet = latest_internet.sort_values("internet_users", ascending=False).head(12)
fig_top = px.bar(
    top_internet.sort_values("internet_users"),
    x="internet_users", y="country", orientation="h",
    color="internet_users", color_continuous_scale="Teal",
    hover_data={"year": True, "iso3": True},
    title="Countries with highest reported internet use",
    labels={"internet_users": "Internet users (%)", "country": "Country"}
)
fig_top.update_layout(template="plotly_white", height=520, coloraxis_showscale=False)
fig_top.show()


## 7. Visualization 3 — Digital connectivity over time

Compare internet use over time for a small set of countries. Change the country list to investigate other economies. Missing years remain missing; lines connect observed values only.

In [ ]:
COUNTRIES_TO_COMPARE = ["USA", "CHN", "IND", "PAK", "BRA", "ZAF", "DEU", "JPN"]
trend = data[data["iso3"].isin(COUNTRIES_TO_COMPARE)].dropna(subset=["internet_users"])
fig_trend = px.line(
    trend, x="year", y="internet_users", color="country",
    markers=True,
    title="Internet use over time — selected countries",
    labels={"year": "Year", "internet_users": "Internet users (%)", "country": "Country"}
)
fig_trend.update_yaxes(range=[0, 100])
fig_trend.update_layout(template="plotly_white", height=540, hovermode="x unified")
fig_trend.show()

## 8. Visualization 4 — Population and economic scale

A bubble chart comparing population with GDP per capita. Bubble size represents population; the logarithmic x-axis helps show countries across very different population scales.

In [ ]:
bubble = latest_by_indicator.dropna(subset=["population", "gdp_per_capita"]).copy()
fig_bubble = px.scatter(
    bubble,
    x="gdp_per_capita",
    y="population",
    size="population",
    color="internet_users",
    hover_name="country",
    hover_data={"gdp_per_capita_year": True, "population_year": True},
    log_x=True,
    log_y=True,
    color_continuous_scale="Teal",
    title="Population scale and GDP per capita",
    labels={
        "gdp_per_capita": "GDP per capita (current US$, log scale)",
        "population": "Population (log scale)",
        "internet_users": "Internet users (%)"
    }
)
fig_bubble.update_layout(template="plotly_white", height=560)
fig_bubble.show()

## 9. Visualization 5 — Digital infrastructure comparison

Compare the latest reported internet use, fixed broadband, and electricity access for selected countries. These measures have different units, so the chart uses separate panels rather than implying that their raw values are directly comparable.

In [ ]:
SELECTED = ["USA", "CHN", "IND", "PAK", "BRA", "ZAF", "DEU", "JPN"]
compare = latest_by_indicator[latest_by_indicator["iso3"].isin(SELECTED)].copy()

long_parts = []
for metric, label in [
    ("internet_users", "Internet users (%)"),
    ("fixed_broadband", "Fixed broadband subscriptions per 100 people"),
    ("electricity_access", "Electricity access (%)")
]:
    part = compare[["country", metric]].rename(columns={metric: "value"}).copy()
    part["indicator"] = label
    long_parts.append(part)

comparison_long = pd.concat(long_parts, ignore_index=True).dropna(subset=["value"])
fig_compare = px.bar(
    comparison_long, x="country", y="value", color="country",
    facet_row="indicator", barmode="group",
    title="Digital infrastructure indicators — selected countries",
    labels={"country": "Country", "value": "Reported indicator value"}
)
fig_compare.update_layout(template="plotly_white", height=780, showlegend=False)
fig_compare.show()

## 10. Interpretation guide

Use the charts to investigate:
1. Are higher-income economies generally more connected?
2. Which countries have improved internet access most over the observed period?
3. Do large populations necessarily correspond to higher or lower connectivity?
4. How do internet use, fixed broadband, and electricity access differ across selected countries?

Avoid causal language. GDP per capita is not a direct measure of AI investment or AI capability. Internet access and infrastructure are enabling conditions, but they do not tell the whole story.

In [ ]:
# Optional: export the processed dataset used for analysis.
data.to_csv("world_bank_digital_readiness.csv", index=False)
latest_by_indicator.to_csv("world_bank_latest_country_profiles.csv", index=False)
print("Exported processed datasets.")

## 11. Sources, definitions, and reproducibility

- World Bank Indicators API: https://api.worldbank.org/
- World Bank Data portal: https://data.worldbank.org/
- Indicator metadata can be checked by replacing the indicator code in:
  `https://api.worldbank.org/v2/indicator/INDICATOR_CODE?format=json`

**Reproducibility:** The API is live. Re-running this notebook later may return revised values or newer observations. Record the run date and retain the downloaded CSV files when submitting the project. Cite the indicator code and reporting year in any presentation.

**Limitations:** Reporting coverage varies by country and indicator; current-US-dollar GDP per capita is affected by exchange rates and inflation; broadband subscriptions are not equivalent to unique people connected; and national averages hide within-country inequality.